# This code gathers publication data from Scopus and Web of Science's API.

Both sources give us access to their APIs for easily gathering publication data. The code below turns the information gathered from the API into Excel files or input directly into a SQLite3 database. 

In [1]:
import pandas as pd
import matplotlib as plt
import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
import requests as req
import re
import json
import pybliometrics
from pybliometrics.scopus import ScopusSearch
from pybliometrics.scopus import SerialTitleSearch
import sqlite3
import clarivate.wos_starter.client
import os
import time
from clarivate.wos_starter.client.rest import ApiException
from pprint import pprint
import xlsxwriter
import requests as req
import re
import json
import warnings
import getpass

warnings.filterwarnings("ignore") #This is here to improve performance time. However, if you are repairing, improving, or otherwise testing this code, it is recommended to comment this out.

# Connect to Publication Database

### If you are planning on turning the publication data into an Excel file instead, you can skip this cell.

In [3]:
conn = sqlite3.connect(r"C:\Users\aharrima\OneDrive - University of Tennessee\Documents\UTK_Publications.db")
cur = conn.cursor()

# Scopus

#### More details on the Scopus package and its functionality can be found here: https://github.com/alistairwalsh/scopus

### The Scopus package uses Scopus' advanced search feature to gather publications, meaning that we need to provide it some search functions instead of keywords. Some of the most important functions for Scopus gathering include:

AF-ID () - The affiliation ID, best method for gathering publications from an institution. The best way to get this info is to go to Scopus directly and search for the institution, which will store the ID in the search history. See below for commonly used AF-IDs.

PUBYEAR - Used for publication year. Treated like a number, so PUBYEAR > X and PUBYEAR = Y are allowed. No parentheses needed.

AUTHLASTNAME() and AUTHFIRSTNAME() - Used for author names. When gathering articles, this will only return the first 100 authors of an article.

REF() - Used for references, such as finding the articles that cite another article.

### Important Affiliation IDs for Scopus Gathering:

The information that goes into AF-ID (  ) when searching for a specific institution. Leave a space before and after the ID itself. For example: AF-ID ( "university" 86753091 )

UT Knoxville - "the university of tennessee, knoxville" 60015574

Montana State - "Montana State University" 60022522

In [33]:
# Initialize the Scopus API Functions

pybliometrics.scopus.init()

### Before running the search, check to see the size of what you are downloading. Large downloads will take several minutes to gather from Scopus.

In [32]:
searchSizeCheck = ScopusSearch('PUBYEAR > 2013 AND PUBYEAR < 2026 AND AF-ID ( "the university of tennessee, knoxville" 60015574 )', download = False)

searchSizeCheck.get_results_size()

46799

### Download Scopus Data

The Scopus package comes with an automatically generated progress bar that includes how long it expects it'll take to gather all of your articles.

In [34]:
scopusPubsSearchUTK = ScopusSearch('PUBYEAR > 2013 AND PUBYEAR < 2026 AND AF-ID ( "the university of tennessee, knoxville" 60015574 )', verbose = True)

100%|██████████| 1872/1872 [38:03<00:00,  1.22s/it]


AttributeError: 'ScopusSearch' object has no attribute 'shape'

### Convert to a DataFrame

In [35]:
scopusPubsUTK = pd.DataFrame(scopusPubsSearchUTK.results)

scopusPubsUTK.columns

Index(['eid', 'doi', 'pii', 'pubmed_id', 'title', 'subtype',
       'subtypeDescription', 'creator', 'afid', 'affilname',
       'affiliation_city', 'affiliation_country', 'author_count',
       'author_names', 'author_ids', 'author_afids', 'coverDate',
       'coverDisplayDate', 'publicationName', 'issn', 'source_id', 'eIssn',
       'aggregationType', 'volume', 'issueIdentifier', 'article_number',
       'pageRange', 'description', 'authkeywords', 'citedby_count',
       'openaccess', 'freetoread', 'freetoreadLabel', 'fund_acr', 'fund_no',
       'fund_sponsor'],
      dtype='object')

In [36]:
scopusPubsUTK.shape

(46799, 36)

## Get the Publication Campus and Year

In [142]:
scopusPubsUTK['Publication Campus'] = 'UT Knoxville'

scopusPubsUTK['Year'] = scopusPubsUTK['coverDate'].str.slice(0, 4)

scopusPubsUTK = scopusPubsUTK.astype({'Year': int})

### Save to Database

Only use if you have already connected to a database in the cell above.

In [143]:
scopusPubsUTK.to_sql('scopus_pubs_utk_update2025', conn, if_exists = 'replace', index = False)

46799

### Save to Excel

Only use if you are not connecting to a database, or if you want a backup to the database.

In [ ]:
scopusPubsUTK.to_excel('Scopus API Download Test.xlsx')

# Web of Science

#### More details on the Web of Science API package and its functionality can be found here: https://github.com/clarivate/wosstarter_python_client?tab=readme-ov-file

As of March 18, 2026, we have access to the Standard Subscriber API, not the Extended API nor the Institutional level.

## Initialize the API and Run the Query

### Similar to Scopus, Web of Science's API uses its advanced search features to gather publications. A table containing all variables you can query on and the format to do so can be found at the above GitHub link.

In [2]:
# Defining the host is optional and defaults to http://api.clarivate.com/apis/wos-starter/v1
# See configuration.py for a list of all supported configuration parameters.
configuration = clarivate.wos_starter.client.Configuration(
    host = "https://api.clarivate.com/apis/wos-starter/v1"
)

# Load the API Key

os.environ["API_KEY"] = getpass.getpass('Enter your Web of Science API Key:')

## Each project should have its own API Key, which can be created from the Web of Science API admin page.


# Configure API key authorization: ClarivateApiKeyAuth
configuration.api_key['ClarivateApiKeyAuth'] = os.environ["API_KEY"]



# Enter a context with an instance of the API client
with clarivate.wos_starter.client.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = clarivate.wos_starter.client.DocumentsApi(api_client)
    query = '(OG=(University of Tennessee Knoxville)) AND PY=(2025)'
    database = 'WOS' # Web of Science Database abbreviation * WOS - Web of Science Core collection * BIOABS - Biological Abstracts * BCI - BIOSIS Citation Index * BIOSIS - BIOSIS Previews * CCC - Current Contents Connect * DIIDW - Derwent Innovations Index * DRCI - Data Citation Index * MEDLINE - MEDLINE The U.S. National Library of Medicine® (NLM®) premier life sciences database. * ZOOREC - Zoological Records * PPRN - Preprint Citation Index * WOK - All databases  (optional) (default to 'WOS')
    recordLimit = 50 # set the limit of records on the page (1-50) (optional) (default to 10)
    resultPage = 1 # set the result page (optional) (default to 1)
    sort_field = 'TC+D' # Order by field(s). Field name and order by clause separated by '+', use A for ASC and D for DESC, ex: PY+D. Multiple values are separated by comma. Supported fields:  * **LD** - Load Date * **PY** - Publication Year * **RS** - Relevance * **TC** - Times Cited  (optional)
    modified_time_span = None # Defines a date range in which the results were most recently modified. Beginning and end dates must be specified in the yyyy-mm-dd format separated by '+' or ' ', e.g. 2023-01-01+2023-12-31. This parameter is not compatible with the all databases search, i.e. db=WOK is not compatible with this parameter. (optional)
    tc_modified_time_span = None # Defines a date range in which times cited counts were modified. Beginning and end dates must be specified in the yyyy-mm-dd format separated by '+' or ' ', e.g. 2023-01-01+2023-12-31. This parameter is not compatible with the all databases search, i.e. db=WOK is not compatible with this parameter. (optional)
    detail = None # it will returns the full data by default, if detail=short it returns the limited data (optional)

    try:
        # Query Web of Science documents 
        api_response = api_instance.documents_get(query, 
                                                  db=database, 
                                                  limit=recordLimit,
                                                  page=resultPage, 
                                                  sort_field=sort_field, 
                                                  modified_time_span=modified_time_span, 
                                                  tc_modified_time_span=tc_modified_time_span, 
                                                  detail=detail)
        #print("The response of DocumentsApi->documents_get:\n")
        #pprint(api_response)
    except ApiException as e:
        print("Exception when calling DocumentsApi->documents_get: %s\n" % e)

Enter your Web of Science API Key: ········


### Check the Query Size

Web of Science will only show up to 50 items by default, as this is the size of one page. However, the full length of the query is shown within the metadata.

If the query is larger than 50 publications, the bulk load section can be used to gather many pages at once.

In [3]:
dictTest = api_response.to_dict()

dictTest['metadata']

{'page': 1, 'limit': 50, 'total': 4652}

# Turn Single Page into Publication Data Table

If each portion of the code shown below is working and you need to gather more than one page's worth of articles (more than 50), scroll down to the bulk load section below.

### uID, Title, Source Type

In [11]:
titleSource = pd.DataFrame.from_dict(dictTest['hits'])[['uid', 'title', 'types', 'sourceTypes']]

titleSource

,uid,title,types,sourceTypes
0,WOS:000272455700005,The Role of Propagule Pressure in Biological I...,"[Review, Book]","[Review, Book Chapter]"
1,WOS:000262615800025,Adolescent Romantic Relationships,"[Review, Book]","[Review, Book Chapter]"
2,WOS:000348451900012,Designing Radiation Resistance in Materials fo...,"[Review, Book]","[Review, Book Chapter]"
3,WOS:000280818700011,The Indentation Size Effect: A Critical Examin...,"[Review, Book]","[Review, Book Chapter]"
4,WOS:000252359800019,Sympatric speciation: Models and empirical evi...,"[Review, Book]","[Review, Book Chapter]"
5,WOS:000329821800013,Community and Ecosystem Responses to Elevation...,"[Review, Book]","[Review, Book Chapter]"
6,WOS:000306933700013,Health Benefits of n-3 Polyunsaturated Fatty A...,"[Article, Book]","[Article, Book Chapter]"
7,WOS:000412189800010,Viruses in Soil Ecosystems: An Unknown Quantit...,"[Article, Book]","[Article, Book Chapter]"
8,WOS:000323371300002,The Appalachian orogen: A brief summary,"[Editorial material, Book]","[Editorial Material, Book Chapter]"
9,WOS:000341798800002,Diversity of <i>Bacillus thuringiensis</i> Cry...,"[Review, Book]","[Review, Book Chapter]"


### Publishing Source, Publishing Date, Page Count

In [25]:
sourceTitle = pd.Series([])
publishYear = pd.Series([])
publishMonth = pd.Series([])
volume = pd.Series([])
issue = pd.Series([])
pages = pd.Series([])

for i in range(len(titleSource)):
    sourceTitle = pd.concat([sourceTitle, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['sourceTitle']]], ignore_index = True)
    publishYear = pd.concat([publishYear, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['publishYear']]], ignore_index = True)
    
    if 'publishMonth' not in pd.DataFrame.from_dict(dictTest['hits'][i]['source']).columns:
        publishMonth = pd.concat([publishMonth, pd.Series(['NA'])], ignore_index = True)
    else:
        publishMonth = pd.concat([publishMonth, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['publishMonth']]], ignore_index = True)

    if 'volume' not in pd.DataFrame.from_dict(dictTest['hits'][i]['source']).columns:
        volume = pd.concat([volume, pd.Series(['NA'])], ignore_index = True)
    else:
        volume = pd.concat([volume, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['volume']]], ignore_index = True)
    
    if 'issue' not in pd.DataFrame.from_dict(dictTest['hits'][i]['source']).columns:
        issue = pd.concat([issue, pd.Series(['NA'])], ignore_index = True)
    else:
        issue = pd.concat([issue, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['issue']]], ignore_index = True)
                           
    pages = pd.concat([pages, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['pages']]], ignore_index = True)

publisherInfo = pd.concat([sourceTitle, publishYear, publishMonth, volume, issue, pages], axis = 1)

publisherInfo = publisherInfo.drop(columns = [0, 1])

#publisherInfo

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49


,sourceTitle,publishYear,volume,pages
0,ANNUAL REVIEW OF ECOLOGY EVOLUTION AND SYSTEMA...,2009.0,40,81-102
1,ANNUAL REVIEW OF PSYCHOLOGY,2009.0,60,631-652
2,"ANNUAL REVIEW OF MATERIALS RESEARCH, VOL 44",2014.0,44,241-267
3,"ANNUAL REVIEW OF MATERIALS RESEARCH, VOL 40",2010.0,40,271-292
4,ANNUAL REVIEW OF ECOLOGY EVOLUTION AND SYSTEMA...,2007.0,38,459-487
5,"ANNUAL REVIEW OF ECOLOGY, EVOLUTION, AND SYSTE...",2013.0,44,261-280
6,"ADVANCES IN FOOD AND NUTRITION RESEARCH, VOL 6...",2012.0,65,211-222
7,"ANNUAL REVIEW OF VIROLOGY, VOL 4",2017.0,4,201-219
8,FROM RODINIA TO PANGEA: THE LITHOTECTONIC RECO...,2010.0,206,1-19
9,INSECT MIDGUT AND INSECTICIDAL PROTEINS,2014.0,47,39-87


### Authors

In [157]:
authorNames = pd.Series([])
authorCount = pd.Series([])

for i in range(len(titleSource)):
    authorCount = pd.concat([authorCount, pd.Series([len(dictTest['hits'][i]['names']['authors'])])], ignore_index = True)

    authorString = ''
    for j in range(len(dictTest['hits'][i]['names']['authors'])):
        authorString = authorString + dictTest['hits'][i]['names']['authors'][j]['displayName'] + '; '

    authorNames = pd.concat([authorNames, pd.Series([authorString])], ignore_index = True)

authors = pd.concat([authorCount, authorNames], axis = 1)

authors.columns = ['authorCount', 'authorNames']

#authors

C:\Users\aharrima\AppData\Local\Temp\ipykernel_51840\2433469683.py:6: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  authorCount = pd.concat([authorCount, pd.Series([len(dictTest['hits'][i]['names']['authors'])])], ignore_index = True)


### Links

In [33]:
recordLink = pd.Series([])
citingArticlesLink = pd.Series([])
referenceLink = pd.Series([])
relatedLink = pd.Series([])

for i in range(len(titleSource)):
    recordLink = pd.concat([recordLink, pd.Series([dictTest['hits'][i]['links']['record']])], ignore_index = True)
    if 'citingArticles' in dictTest['hits'][i]['links']:
        citingArticlesLink = pd.concat([citingArticlesLink, pd.Series([dictTest['hits'][i]['links']['citingArticles']])], ignore_index = True)
    else:
        citingArticlesLink = pd.concat([citingArticlesLink, pd.Series(['None'])], ignore_index = True)
    if 'references' in dictTest['hits'][i]['links']:
        referenceLink = pd.concat([referenceLink, pd.Series([dictTest['hits'][i]['links']['references']])], ignore_index = True)
    else:
        referenceLink = pd.concat([referenceLink, pd.Series(['None'])], ignore_index = True)
    if 'related' in dictTest['hits'][i]['links']:
        relatedLink = pd.concat([relatedLink, pd.Series([dictTest['hits'][i]['links']['related']])], ignore_index = True)
    else:
        relatedLink = pd.concat([relatedLink, pd.Series(['None'])], ignore_index = True)

links = pd.concat([recordLink, citingArticlesLink, referenceLink, relatedLink], axis = 1)

links.columns = ['recordLink', 'citingArticlesLink', 'referenceLink', 'relatedLink']

#links

### Identifiers and Citations

In [37]:
citationCount = pd.Series([])
doi = pd.Series([])
ISSN = pd.Series([])
keywords = pd.Series([])

for i in range(len(titleSource)):
    citationCount = pd.concat([citationCount, pd.Series([dictTest['hits'][i]['citations'][0]['count']])], ignore_index = True)

    if 'doi' in dictTest['hits'][i]['identifiers']:
        doi = pd.concat([doi, pd.Series([dictTest['hits'][i]['identifiers']['doi']])], ignore_index = True)
    else:
        doi = pd.concat([doi, pd.Series(['NA'])], ignore_index = True)
    
    if 'eissn' in dictTest['hits'][i]['identifiers']:
        ISSN = pd.concat([ISSN, pd.Series([dictTest['hits'][i]['identifiers']['eissn']])], ignore_index = True)
    elif 'issn' in dictTest['hits'][i]['identifiers'] and 'eissn' not in dictTest['hits'][i]['identifiers']:
        ISSN = pd.concat([ISSN, pd.Series([dictTest['hits'][i]['identifiers']['issn']])], ignore_index = True)
    else:
        ISSN = pd.concat([ISSN, pd.Series(['NA'])], ignore_index = True)
        
    keywords = pd.concat([keywords, pd.Series([dictTest['hits'][i]['keywords']['authorKeywords']])], ignore_index = True)

idCite = pd.concat([citationCount, doi, ISSN, keywords], axis = 1)

idCite.columns = ['citationCount', 'doi', 'ISSN', 'keywords']

#idCite

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49


C:\Users\aharrima\AppData\Local\Temp\ipykernel_63400\1588619565.py:8: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  citationCount = pd.concat([citationCount, pd.Series([dictTest['hits'][i]['citations'][0]['count']])], ignore_index = True)


### Combine all Parts Together

In [160]:
fullPublicationInfo = pd.concat([titleSource, publisherInfo, authors, links, idCite], axis = 1)

fullPublicationInfo

,uid,title,types,sourceTypes,sourceTitle,publishYear,publishMonth,volume,issue,pages,authorCount,authorNames,recordLink,citingArticlesLink,referenceLink,relatedLink,citationCount,doi,ISSN,keywords
0,WOS:001543361000014,The microbiome diversifies long-to short-chain...,[Article],[Article],CELL,2025.0,JUL 24,188,15,4154-4169,48,"Mannochio-Russo, Helena; Charron-Lamoureux, Vi...",https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,11,10.1016/j.cell.2025.05.015,1097-4172,[]
1,WOS:001373010000001,Scientific exploration with expert knowledge (...,[Article],[Article],DIGITAL DISCOVERY,2025.0,JAN 15,4,1,252-263,5,"Pratiush, Utkarsh; Funakubo, Hiroshi; Vasudeva...",https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,8,10.1039/d4dd00277f,2635-098X,[]
2,WOS:001369650000001,Advancements of PAPI for the exascale generation,[Article],[Article],INTERNATIONAL JOURNAL OF HIGH PERFORMANCE COMP...,2025.0,MAR,39,2,251-268,6,"Jagode, Heike; Danalis, Anthony; Congiu, Giuse...",https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,5,10.1177/10943420241303884,1741-2846,"[Performance monitoring, power monitoring, ene..."
3,WOS:001468571900001,Single-Ion Conducting Polymer Electrolyte Enab...,[Article],[Article],MACROMOLECULES,2025.0,APR 16,58,9,4708-4714,6,"Lehmann, Michelle L.; Ock, Jiyoung; Gainaru, C...",https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,5,10.1021/acs.macromol.5c00422,1520-5835,[]
4,WOS:001476711300001,AI Chatbots and Subject Cataloging A Performan...,[Article],[Article],LIBRARY RESOURCES & TECHNICAL SERVICES,2025.0,APR,69,2,14,2,"Dobreski, Brian; Hastings, Christopher;",https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,2,10.5860/lrts.69n2.8440,2159-9610,[]
5,WOS:001451276700005,Student Engagement in Academic Libraries: A Co...,[Article],[Article],COLLEGE & RESEARCH LIBRARIES,2025.0,MAR,86,2,275-293,4,"Zhu, Xiaohua Awa; Whitaker, Erin E.; Cho, Moon...",https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,3,10.5860/crl.86.2.275,2150-6701,[]
6,WOS:001324301600001,Evolution of the SLATE linear algebra library,[Article],[Article],INTERNATIONAL JOURNAL OF HIGH PERFORMANCE COMP...,2025.0,JAN,39,1,3-17,11,"Gates, Mark; Abdelfattah, Ahmad; Akbudak, Kadi...",https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,3,10.1177/10943420241286531,1741-2846,"[High-performance computing, graphics processi..."
7,WOS:001453134100001,Battle Royale Optimization for Optimal Band Se...,[Article],[Article],JOURNAL OF IMAGING,2025.0,MAR 17,11,3,21,7,"Ramasamy, Jagadeeswaran; Raju, Anand; Ranganat...",https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,https://www.webofscience.com/api/gateway?GWVer...,2,10.3390/jimaging11030083,2313-433X,"[spectral techniques, soil properties, machine..."
8,WOS:001428592200001,SANE: strategic autonomous non-smooth explorat...,[Article],[Article],DIGITAL DISCOVERY,2025.0,MAR 12,4,3,853-867,6,"Biswas, Arpan; Vasudevan

### Export the Page to Excel

In [162]:
fullPublicationInfo.to_excel('API Test Run Web of Science.xlsx', index = False)

# Web of Science Bulk Gathering 

This part of the code combines every piece seen above into one cell, so many pages can be gathered at one time. If you are unsure how many items fit your query, go to the "Check the Query Size" cell above. 


## NOTE: Web of Science caps each person's daily publication gathering at 5,000 items. If you are gathering more than 5000 items, you will need to spread out the gathering over multiple days or get it from the WoS website.

In [13]:
allQuery = pd.DataFrame()

configuration = clarivate.wos_starter.client.Configuration(
    host = "https://api.clarivate.com/apis/wos-starter/v1"
)

# Load the API Key

os.environ["API_KEY"] = getpass.getpass('Enter your Web of Science API Key:')

# Configure API key authorization: ClarivateApiKeyAuth
configuration.api_key['ClarivateApiKeyAuth'] = os.environ["API_KEY"]


# Set the number of pages needed for your query here

numPages = 24

for page in range(1, numPages + 1):
    print(r'Page '+str(page)+' Gathering...')
    with clarivate.wos_starter.client.ApiClient(configuration) as api_client:
        # Create an instance of the API class
        api_instance = clarivate.wos_starter.client.DocumentsApi(api_client)
        query = '(OG=(University of Tennessee Knoxville)) AND PY=(2026)'
        database = 'WOS' # Web of Science Database abbreviation * WOS - Web of Science Core collection * BIOABS - Biological Abstracts * BCI - BIOSIS Citation Index * BIOSIS - BIOSIS Previews * CCC - Current Contents Connect * DIIDW - Derwent Innovations Index * DRCI - Data Citation Index * MEDLINE - MEDLINE The U.S. National Library of Medicine® (NLM®) premier life sciences database. * ZOOREC - Zoological Records * PPRN - Preprint Citation Index * WOK - All databases  (optional) (default to 'WOS')
        recordLimit = 50 # set the limit of records on the page (1-50) (optional) (default to 10)
        resultPage = page # set the result page (optional) (default to 1)
        sort_field = 'TC+D' # Order by field(s). Field name and order by clause separated by '+', use A for ASC and D for DESC, ex: PY+D. Multiple values are separated by comma. Supported fields:  * **LD** - Load Date * **PY** - Publication Year * **RS** - Relevance * **TC** - Times Cited  (optional)
        modified_time_span = None # Defines a date range in which the results were most recently modified. Beginning and end dates must be specified in the yyyy-mm-dd format separated by '+' or ' ', e.g. 2023-01-01+2023-12-31. This parameter is not compatible with the all databases search, i.e. db=WOK is not compatible with this parameter. (optional)
        tc_modified_time_span = None # Defines a date range in which times cited counts were modified. Beginning and end dates must be specified in the yyyy-mm-dd format separated by '+' or ' ', e.g. 2023-01-01+2023-12-31. This parameter is not compatible with the all databases search, i.e. db=WOK is not compatible with this parameter. (optional)
        detail = None # it will returns the full data by default, if detail=short it returns the limited data (optional)
    
        try:
            # Query Web of Science documents 
            api_response = api_instance.documents_get(query, 
                                                      db=database, 
                                                      limit=recordLimit,
                                                      page=resultPage, 
                                                      sort_field=sort_field, 
                                                      modified_time_span=modified_time_span, 
                                                      tc_modified_time_span=tc_modified_time_span, 
                                                      detail=detail)
            #print("The response of DocumentsApi->documents_get:\n")
            #pprint(api_response)
        except ApiException as e:
            print("Exception when calling DocumentsApi->documents_get: %s\n" % e)
            break

        # Convert to Dictionary
        dictTest = api_response.to_dict()

        
        # uID, Title, Source Type
        titleSource = pd.DataFrame.from_dict(dictTest['hits'])[['uid', 'title', 'types', 'sourceTypes']]

        
        # Publishing Source, Publishing Date, Page Count
        sourceTitle = pd.Series([])
        publishYear = pd.Series([])
        publishMonth = pd.Series([])
        volume = pd.Series([])
        issue = pd.Series([])
        pages = pd.Series([])
        
        for i in range(len(titleSource)):
            sourceTitle = pd.concat([sourceTitle, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['sourceTitle']]], ignore_index = True)
            publishYear = pd.concat([publishYear, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['publishYear']]], ignore_index = True)
            
            if 'publishMonth' not in pd.DataFrame.from_dict(dictTest['hits'][i]['source']).columns:
                publishMonth = pd.concat([publishMonth, pd.Series(['NA'])], ignore_index = True)
            else:
                publishMonth = pd.concat([publishMonth, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['publishMonth']]], ignore_index = True)
        
            if 'volume' not in pd.DataFrame.from_dict(dictTest['hits'][i]['source']).columns:
                volume = pd.concat([volume, pd.Series(['NA'])], ignore_index = True)
            else:
                volume = pd.concat([volume, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['volume']]], ignore_index = True)
            
            if 'issue' in pd.DataFrame.from_dict(dictTest['hits'][i]['source']).columns:
                issue = pd.concat([issue, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['issue']]], ignore_index = True)
            else:
                issue = pd.concat([issue, pd.Series(['NA'])], ignore_index = True)
                                   
            pages = pd.concat([pages, pd.DataFrame.from_dict(dictTest['hits'][i]['source']).head(1)[['pages']]], ignore_index = True)
        
        publisherInfo = pd.concat([sourceTitle, publishYear, publishMonth, volume, issue, pages], axis = 1)

        if 0 in publisherInfo.columns and 1 in publisherInfo.columns:
            publisherInfo = publisherInfo = publisherInfo.drop(columns = [0, 1])
        elif 0 in publisherInfo.columns:
            publisherInfo = publisherInfo = publisherInfo.drop(0, axis = 1)
        elif 1 in publisherInfo.columns:
            publisherInfo = publisherInfo = publisherInfo.drop(1, axis = 1)
        else:
            publisherInfo = publisherInfo


        # Authors
        authorNames = pd.Series([])
        authorCount = pd.Series([])
        
        
        for i in range(len(titleSource)):
            authorCount = pd.concat([authorCount, pd.Series([len(dictTest['hits'][i]['names']['authors'])])], ignore_index = True)
        
            authorString = ''
            for j in range(len(dictTest['hits'][i]['names']['authors'])):
                authorString = authorString + dictTest['hits'][i]['names']['authors'][j]['displayName'] + '; '
        
            authorNames = pd.concat([authorNames, pd.Series([authorString])], ignore_index = True)
        
        authors = pd.concat([authorCount, authorNames], axis = 1)
        
        authors.columns = ['authorCount', 'authorNames']


        # Links
        recordLink = pd.Series([])
        citingArticlesLink = pd.Series([])
        referenceLink = pd.Series([])
        relatedLink = pd.Series([])
        
        for i in range(len(titleSource)):
            recordLink = pd.concat([recordLink, pd.Series([dictTest['hits'][i]['links']['record']])], ignore_index = True)
            if 'citingArticles' in dictTest['hits'][i]['links']:
                citingArticlesLink = pd.concat([citingArticlesLink, pd.Series([dictTest['hits'][i]['links']['citingArticles']])], ignore_index = True)
            else:
                citingArticlesLink = pd.concat([citingArticlesLink, pd.Series(['None'])], ignore_index = True)
            if 'references' in dictTest['hits'][i]['links']:
                referenceLink = pd.concat([referenceLink, pd.Series([dictTest['hits'][i]['links']['references']])], ignore_index = True)
            else:
                referenceLink = pd.concat([referenceLink, pd.Series(['None'])], ignore_index = True)
            if 'related' in dictTest['hits'][i]['links']:
                relatedLink = pd.concat([relatedLink, pd.Series([dictTest['hits'][i]['links']['related']])], ignore_index = True)
            else:
                relatedLink = pd.concat([relatedLink, pd.Series(['None'])], ignore_index = True)
        
        links = pd.concat([recordLink, citingArticlesLink, referenceLink, relatedLink], axis = 1)
        
        links.columns = ['recordLink', 'citingArticlesLink', 'referenceLink', 'relatedLink']


        # Identifiers and Citations
        citationCount = pd.Series([])
        doi = pd.Series([])
        ISSN = pd.Series([])
        keywords = pd.Series([])
        
        for i in range(len(titleSource)):
            citationCount = pd.concat([citationCount, pd.Series([dictTest['hits'][i]['citations'][0]['count']])], ignore_index = True)
            
            if 'doi' in dictTest['hits'][i]['identifiers']:
                doi = pd.concat([doi, pd.Series([dictTest['hits'][i]['identifiers']['doi']])], ignore_index = True)
            else:
                doi = pd.concat([doi, pd.Series(['NA'])], ignore_index = True)
                    
            if 'eissn' in dictTest['hits'][i]['identifiers']:
                ISSN = pd.concat([ISSN, pd.Series([dictTest['hits'][i]['identifiers']['eissn']])], ignore_index = True)
            elif 'issn' in dictTest['hits'][i]['identifiers'] and 'eissn' not in dictTest['hits'][i]['identifiers']:
                ISSN = pd.concat([ISSN, pd.Series([dictTest['hits'][i]['identifiers']['issn']])], ignore_index = True)
            else:
                ISSN = pd.concat([ISSN, pd.Series(['NA'])], ignore_index = True)
                
            keywords = pd.concat([keywords, pd.Series([dictTest['hits'][i]['keywords']['authorKeywords']])], ignore_index = True)
        
        idCite = pd.concat([citationCount, doi, ISSN, keywords], axis = 1)
        
        idCite.columns = ['citationCount', 'doi', 'ISSN', 'keywords']


        # All Columns
        fullPublicationInfo = pd.concat([titleSource, publisherInfo, authors, links, idCite], axis = 1)

    allQuery = pd.concat([allQuery, fullPublicationInfo], ignore_index = True)

allQuery.shape

Page 1 Gathering...
Page 2 Gathering...
Page 3 Gathering...
Page 4 Gathering...
Page 5 Gathering...
Page 6 Gathering...
Page 7 Gathering...
Page 8 Gathering...
Page 9 Gathering...
Page 10 Gathering...
Page 11 Gathering...
Page 12 Gathering...
Page 13 Gathering...
Page 14 Gathering...
Page 15 Gathering...
Page 16 Gathering...
Page 17 Gathering...
Page 18 Gathering...
Page 19 Gathering...
Page 20 Gathering...
Page 21 Gathering...
Page 22 Gathering...
Page 23 Gathering...
Page 24 Gathering...


(1152, 20)

In [14]:
allQuery.to_excel('UT Publishing Web of Science 2026.xlsx', index = False)

# Next Steps:

### Looking to gather OpenAlex publishing data?

Check out the "Gathering OpenAlex Articles Using API" R code.

### Looking to combine the data you just gathered together?

The "UT Summary of Publications", "Peer Institution Summary of Publications", and/or "UT System Publication Joining" files show how to join publishing data in various ways. These are designed with the full record in mind, so they may need some edits for the API data.

### Does the API not provide the detail you are looking for?

You may have to go to the websites and directly download publishing data from there. Both websites allow for the same searching method used here, so you can keep the process consistent.

### Looking to gather Dimensions or Academic Analytics publishing data?

Neither Dimensions nor Academic Analytics have an API of any description, so you'll need to download that data from the websites directly.

## Publisher Name from Scopus Test Code 

This code does work, but is very slow. It is not recommended for very large collections of articles from Scopus.

In [ ]:
publicationNameFrame = pd.DataFrame()

for i in range(100):#len(scopusPubsUTK['publicationName'])):
    if i % 10 == 0:
        print(i)
    if i != 0:
        titlesFound = set(publicationNameFrame['title'].unique())
        
    serialQuery = SerialTitleSearch(query = {"title": scopusPubsUTK['publicationName'].iloc[i]}, view = 'STANDARD')
    if serialQuery.get_results_size() == 0:
        print('No items found')
        continue
    else:
        serialTest = pd.DataFrame(serialQuery.results)
        if publicationNameFrame.shape[0] == 0:
            serialTest = serialTest[['title', 'publisher', 'subject_area_names']]
            publicationNameFrame = serialTest
            continue
        elif serialTest['title'].iloc[0] in set(titlesFound):
            print('Title Already in List')
            continue
        else:
            serialTest = serialTest[['title', 'publisher', 'subject_area_names']]
            publicationNameFrame = pd.concat([publicationNameFrame, serialTest], ignore_index=True)